# StudyMate: освітній асистент з точних наук

**Фінальний проєкт курсу AI Fundamental**
**Автор:** Яковенко Сергій

---

## Одним абзацом

StudyMate допомагає студенту, який готується до іспиту сам, знайти не просто формулу,
а **правильну для його ситуації** формулу. Ключова властивість системи не в тому,
що вона багато знає, а в тому, що вона **не вигадує**: усі факти приходять з перевіреної
бази, придатність формули перевіряє код, а коли даних немає, система про це прямо каже.

## Чому саме так

Цей принцип не взятий з підручника, я прийшов до нього через два власні результати.

**Експеримент з ембеддінгами.** Я чисельно перевірив, як модель бачить два речення:
«формула працює лише коли точка кидання і точка падіння на одній висоті» і «камінь
кидають з даху, тому початкова висота не дорівнює нулю». Логічно це пряма суперечність,
друге описує ситуацію, у якій перше забороняє застосовувати формулу. Для моделі вони
просто близькі за темою. **Ембеддінг кодує тему, а не істинність.**

**Баг у власному пошуку.** Перша версія ранжування формул була односторонньою мірою:
скільки слів назви знайшлося в запиті. На запиті «закон збереження енергії» вона
повернула **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг. Причина дрібна: слово
«ома» коротке, фільтр його викидав, у назві лишалося одне слово «закон», воно в запиті є.

Обидва випадки це одна й та сама помилка: **впевнена неправильна відповідь**. Для
освітнього продукту вона небезпечніша за відмову, бо студент звернувся саме тому,
що не може її перевірити. Уся архітектура нижче побудована навколо цього.

## Крок 0. Середовище

In [ ]:
!pip install --quiet "langchain>=1.0" "langchain-openai>=1.0" langgraph pandas numpy

In [ ]:
import os

# Ключ беремо з Colab Secrets, далі зі змінної середовища, і лише потім питаємо вручну.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("✅ Ключ завантажено з Colab Secrets")
    except Exception:
        import getpass

        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
        print("✅ Ключ встановлено вручну")
else:
    print("✅ Ключ узято зі змінної середовища")

In [ ]:
import json
import math
import re
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

LLM_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("✅ Імпорти готові")

## Крок 1. Дані: база формул з умовами застосовності

Це серце системи. Кожна картка несе не лише формулу, а три речі, яких немає
ні в пошуковій видачі, ні в пам'яті моделі:

- **`predicates`**: машиночитані умови застосовності. Саме за ними код, а не модель,
  вирішує, чи можна застосувати формулу в конкретній задачі.
- **`синоніми`**: те, як формулу називає студент, а не довідник. Закриває лексичний
  розрив між «концентрація розчину» і «молярна концентрація».
- **`примітка`**: попередження людською мовою, яке обов'язково доходить до студента.

In [ ]:
@dataclass(frozen=True)
class Formula:
    """Картка формули. Незмінна: дані не мають правитися під час роботи системи."""

    uid: str
    subject: str
    name: str
    expression: str
    variables: dict
    example: str
    synonyms: tuple = ()
    predicates: tuple = ()      # машиночитані умови застосовності
    note: str = ""              # попередження для студента
    source: str = ""


FORMULAS = [
    # --- фізика: кінематика -------------------------------------------------
    Formula(
        uid="phys.projectile.range_flat",
        subject="фізика",
        name="дальність польоту тіла, кинутого під кутом",
        expression="L = v₀² · sin(2α) / g",
        variables={
            "L": "дальність польоту (м)",
            "v₀": "початкова швидкість (м/с)",
            "α": "кут кидання до горизонту",
            "g": "прискорення вільного падіння (9.8 м/с²)",
        },
        example="v₀ = 20 м/с, α = 45°: L = 400 · 1 / 9.8 ≈ 40.8 м",
        synonyms=("дальність кидка", "куди впаде тіло", "на яку відстань полетить"),
        predicates=("h0 == 0", "air_resistance == False"),
        note="Формула виведена для випадку, коли точка кидання і точка падіння на одній "
             "висоті. Для кидання з даху, вежі чи столу вона занижує відповідь.",
        source="Загальна фізика, розділ 2.4",
    ),
    Formula(
        uid="phys.projectile.range_elevated",
        subject="фізика",
        name="дальність польоту при киданні з висоти",
        expression="час t з рівняння h₀ + v₀·sin(α)·t − g·t²/2 = 0, далі L = v₀·cos(α)·t",
        variables={
            "h₀": "початкова висота (м)",
            "v₀": "початкова швидкість (м/с)",
            "α": "кут кидання",
            "t": "час польоту (с)",
        },
        example="h₀ = 12 м, v₀ = 15 м/с, α = 30°: t ≈ 2.51 с, L ≈ 32.5 м",
        synonyms=("кидання з даху", "кидання з висоти", "кинули з балкона"),
        predicates=("h0 > 0", "air_resistance == False"),
        note="Саме цей випадок плутають з формулою для рівної поверхні найчастіше.",
        source="Загальна фізика, розділ 2.5",
    ),
    Formula(
        uid="phys.energy.kinetic",
        subject="фізика",
        name="кінетична енергія",
        expression="Eₖ = m·v² / 2",
        variables={"Eₖ": "енергія (Дж)", "m": "маса (кг)", "v": "швидкість (м/с)"},
        example="m = 2 кг, v = 3 м/с: Eₖ = 9 Дж",
        synonyms=("енергія руху",),
        predicates=("v_units == 'м/с'",),
        note="Швидкість обов'язково в м/с. Якщо дано км/год, спершу переведи одиниці.",
        source="Загальна фізика, розділ 3.1",
    ),
    Formula(
        uid="phys.energy.potential",
        subject="фізика",
        name="потенціальна енергія",
        expression="Eₚ = m·g·h",
        variables={"Eₚ": "енергія (Дж)", "m": "маса (кг)", "g": "9.8 м/с²", "h": "висота (м)"},
        example="m = 5 кг, h = 10 м: Eₚ = 490 Дж",
        synonyms=("енергія висоти",),
        note="Висота відлічується від рівня, який ти сам обрав за нульовий.",
        source="Загальна фізика, розділ 3.2",
    ),
    Formula(
        uid="phys.electricity.ohm",
        subject="фізика",
        name="закон ома",
        expression="I = U / R",
        variables={"I": "сила струму (А)", "U": "напруга (В)", "R": "опір (Ом)"},
        example="U = 12 В, R = 4 Ом: I = 3 А",
        synonyms=("сила струму через напругу",),
        predicates=("resistance_constant == True",),
        note="Виконується для ділянки кола з постійним опором.",
        source="Загальна фізика, розділ 5.2",
    ),
    Formula(
        uid="phys.kinematics.speed",
        subject="фізика",
        name="середня швидкість",
        expression="v = s / t",
        variables={"v": "швидкість (м/с)", "s": "шлях (м)", "t": "час (с)"},
        example="s = 100 м, t = 10 с: v = 10 м/с",
        note="Це середня швидкість. Для змінного руху миттєва швидкість інша.",
        source="Загальна фізика, розділ 1.2",
    ),
    # --- хімія --------------------------------------------------------------
    Formula(
        uid="chem.gas.ideal",
        subject="хімія",
        name="рівняння стану ідеального газу",
        expression="p·V = n·R·T",
        variables={
            "p": "тиск (Па)", "V": "об'єм (м³)", "n": "кількість речовини (моль)",
            "R": "8.314 Дж/(моль·К)", "T": "температура (К)",
        },
        example="p = 1.2 МПа, V = 0.04 м³, T = 300 К: n = pV/(RT) ≈ 19.5 моль",
        synonyms=("рівняння менделєєва клапейрона", "закон ідеального газу"),
        predicates=("T_units == 'К'", "V_units == 'м³'"),
        note="Температура обов'язково в кельвінах, об'єм у м³. Найчастіша помилка: "
             "40 л це 0.04 м³, а не 0.4 м³.",
        source="Загальна хімія, розділ 4.1",
    ),
    Formula(
        uid="chem.solution.molar_concentration",
        subject="хімія",
        name="молярна концентрація",
        expression="C = n / V",
        variables={"C": "концентрація (моль/л)", "n": "кількість (моль)", "V": "об'єм розчину (л)"},
        example="0.5 моль у 2 л: C = 0.25 моль/л",
        synonyms=("концентрація розчину", "концентрація речовини"),
        note="V це об'єм усього розчину, а не лише розчинника.",
        source="Загальна хімія, розділ 6.3",
    ),
    Formula(
        uid="chem.molar_mass",
        subject="хімія",
        name="молярна маса",
        expression="M = m / n",
        variables={"M": "молярна маса (г/моль)", "m": "маса (г)", "n": "кількість (моль)"},
        example="36 г води, 2 моль: M = 18 г/моль",
        source="Загальна хімія, розділ 2.1",
    ),
    # --- математика ---------------------------------------------------------
    Formula(
        uid="math.geometry.circle_area",
        subject="математика",
        name="площа кола",
        expression="S = π·r²",
        variables={"S": "площа", "r": "радіус"},
        example="r = 5 см: S ≈ 78.54 см²",
        source="Геометрія, розділ 8",
    ),
    Formula(
        uid="math.geometry.pythagoras",
        subject="математика",
        name="теорема піфагора",
        expression="a² + b² = c²",
        variables={"a, b": "катети", "c": "гіпотенуза"},
        example="a = 3, b = 4: c = 5",
        synonyms=("гіпотенуза", "сторони прямокутного трикутника"),
        predicates=("triangle_type == 'прямокутний'",),
        note="Працює лише для прямокутних трикутників.",
        source="Геометрія, розділ 5",
    ),
    Formula(
        uid="math.algebra.quadratic",
        subject="математика",
        name="квадратне рівняння",
        expression="x = (−b ± √(b² − 4ac)) / 2a",
        variables={"a, b, c": "коефіцієнти рівняння ax² + bx + c = 0"},
        example="x² − 5x + 6 = 0: x₁ = 3, x₂ = 2",
        synonyms=("дискримінант", "корені рівняння"),
        note="Дискримінант D = b² − 4ac визначає кількість коренів.",
        source="Алгебра, розділ 7",
    ),
]

FORMULAS_BY_UID = {f.uid: f for f in FORMULAS}

SUBJECTS = {}
for _formula in FORMULAS:
    SUBJECTS.setdefault(_formula.subject, []).append(_formula)

STATS = {
    subject: (len(items), sum(1 for f in items if f.predicates))
    for subject, items in SUBJECTS.items()
}

print(f"✅ База формул: {len(FORMULAS)} карток")
print("   " + " | ".join(f"{s}: {n} формул, з умовами {p}" for s, (n, p) in STATS.items()))
print(f"   Синонімів усього: {sum(len(f.synonyms) for f in FORMULAS)}")

## Крок 2. Retrieval: гібридний пошук

Пошук зроблено гібридним свідомо, і кожна половина закриває свій тип розриву.

**Лексичний шар** працює завжди, без мережі й без ключа. Він ловить точні назви,
символи формул і синоніми. Метрика тут не випадкова: це **двостороння міра F2**,
де повнота покриття запиту важить більше за покриття назви. Односторонній варіант
я вже пробував, і саме він повертав ЗАКОН ОМА на запит про збереження енергії.

**Семантичний шар** вмикається за наявності ключа і ловить те, чого не ловить
лексика: перефразування, побутову мову, іншу мову. Це прямий висновок з мого
експерименту з ембеддінгами.

**Злиття через RRF** (reciprocal rank fusion) не потребує калібрування шкал:
складаються не оцінки, а обернені ранги, тому шари не треба зводити до спільної шкали.

In [ ]:
STOP_WORDS = {
    "формула", "формулу", "формули", "яка", "який", "яке", "що", "таке", "як",
    "мені", "потрібна", "потрібно", "розкажи", "поясни", "для", "про", "будь",
    "ласка", "підкажи", "напиши", "покажи", "треба", "хочу", "знайти",
}
MIN_WORD_LENGTH, STEM_LENGTH = 3, 5
LEXICAL_THRESHOLD = 0.6      # нижче цього збіг вважається слабким
RRF_K = 60                   # стандартна константа згладжування для RRF


def stem_word(word: str) -> str:
    """Корінь слова, стійкий до українських відмінків.

    Просте обрізання до N символів тут не працює: «площа» і «площі» мають однакову
    довжину 5, тому обидва лишалися б собою і не збігалися. Тому коротким словам
    відрізаємо принаймні одну літеру закінчення, а довгі ріжемо до STEM_LENGTH.
    «маса»/«маси» дають «мас», «площа»/«площі» дають «площ», «енергія»/«енергії»
    дають «енерг».
    """
    if len(word) > STEM_LENGTH:
        return word[:STEM_LENGTH]
    if len(word) >= 4:
        return word[:len(word) - 1]
    return word


def stems(text: str, drop_stop: bool = False) -> set:
    """Множина коренів значущих слів."""
    words = [w.strip(".,;:!?()«»\"'") for w in text.lower().split()]
    return {
        stem_word(w) for w in words
        if len(w) >= MIN_WORD_LENGTH and not (drop_stop and w in STOP_WORDS)
    }


def lexical_score(query: str, target: str) -> float:
    """Двостороння міра схожості F2: покриття запиту важить більше за покриття назви.

    Чому саме так, а не проста частка збігів. Назва «швидкість» складається з одного
    слова, тому запит «швидкість світла» покриває її на 100%, і симетрична міра
    вважала б це відмінним збігом, хоча про світло в базі немає нічого. Слово запиту,
    якого немає в назві, це сигнал «питають не про це», і він має важити сильніше.
    """
    q, t = stems(query, drop_stop=True), stems(target)
    if not q or not t:
        return 0.0
    hits = len(q & t)
    if not hits:
        return 0.0
    precision, recall = hits / len(t), hits / len(q)
    beta_sq = 4
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


def lexical_search(query: str, top_k: int = 5) -> list:
    """Лексичний пошук по назвах і синонімах. Повертає [(uid, score), ...]."""
    scored = []
    for formula in FORMULAS:
        best = max(
            [lexical_score(query, formula.name)]
            + [lexical_score(query, syn) for syn in formula.synonyms]
        )
        if best > 0:
            scored.append((formula.uid, best))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]


print("✅ Лексичний шар готовий")

In [ ]:
class SemanticSearch:
    """Семантичний шар на ембеддінгах OpenAI.

    Вмикається лише за наявності ключа. Якщо ключа немає або API недоступне,
    система не падає, а працює на самому лексичному шарі: демо має відтворюватися
    в будь-якому середовищі, а деградація має бути явною, а не тихою.
    """

    def __init__(self, formulas: Sequence[Formula], model: str = EMBED_MODEL):
        self.formulas = list(formulas)
        self.model = model
        self.available = False
        self.vectors = None
        self.tokens_used = 0

    def build(self) -> bool:
        """Будує індекс. Повертає True, якщо семантичний шар доступний."""
        try:
            from openai import OpenAI

            self._client = OpenAI()
            # Для ембеддінга беремо назву, синоніми і примітку: саме вони несуть зміст,
            # а не сам вираз формули, який складається з символів.
            texts = [
                f"{f.name}. {' '.join(f.synonyms)}. {f.note}".strip()
                for f in self.formulas
            ]
            response = self._client.embeddings.create(model=self.model, input=texts)
            self.tokens_used += response.usage.total_tokens
            self.vectors = np.array([item.embedding for item in response.data])
            self.available = True
            print(f"✅ Семантичний шар побудовано: {self.vectors.shape}, "
                  f"токенів {response.usage.total_tokens}")
        except Exception as error:
            print(f"⚠️  Семантичний шар вимкнено ({type(error).__name__}). "
                  "Система працює на лексичному пошуку.")
            self.available = False
        return self.available

    def search(self, query: str, top_k: int = 5) -> list:
        """Пошук за змістом. Порожній список, якщо шар недоступний."""
        if not self.available:
            return []
        try:
            response = self._client.embeddings.create(model=self.model, input=[query])
            self.tokens_used += response.usage.total_tokens
            q = np.array(response.data[0].embedding)
            # Вектори OpenAI нормалізовані, тому косинус це звичайний скалярний добуток.
            sims = self.vectors @ q
            order = np.argsort(sims)[::-1][:top_k]
            return [(self.formulas[i].uid, float(sims[i])) for i in order]
        except Exception:
            return []


semantic = SemanticSearch(FORMULAS)
# У ноутбуці будуємо одразу, щоб демо працювало «з коробки».
# Модуль studymate_core робить це ліниво, з боку застосунку.
semantic.build()

In [ ]:
def reciprocal_rank_fusion(*rankings: list, k: int = RRF_K) -> list:
    """Зливає кілька ранжувань у одне за формулою RRF: score = Σ 1/(k + rank).

    Перевага перед складанням оцінок у тому, що шкали шарів не треба зводити
    докупи: лексична міра лежить у [0,1], косинусна близькість теж, але розподілені
    вони по-різному. RRF працює з порядком, а не зі значеннями.
    """
    scores = {}
    for ranking in rankings:
        for rank, (uid, _) in enumerate(ranking, start=1):
            scores[uid] = scores.get(uid, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda pair: pair[1], reverse=True)


@dataclass
class RetrievalResult:
    """Результат пошуку з поясненням, звідки він узявся і наскільки йому можна вірити."""

    formula: Formula
    score: float
    found_by: str            # lexical, semantic або both
    lexical_score: float = 0.0
    confident: bool = False  # чи достатньо збігу, щоб віддати єдину відповідь


def hybrid_search(query: str, top_k: int = 3) -> list:
    """Гібридний пошук: лексика плюс семантика, злиття через RRF.

    Кожен результат несе прапорець `confident`. Він потрібен тому, що сам факт
    «щось знайшлося» нічого не означає: слабкий збіг за одним загальним словом
    теж потрапляє у видачу. Рішення «віддати відповідь чи перепитати» ухвалюється
    за цим прапорцем, а не за фактом наявності результату.
    """
    lex = lexical_search(query, top_k=5)
    sem = semantic.search(query, top_k=5)

    lex_uids = {uid for uid, _ in lex}
    sem_uids = {uid for uid, _ in sem}
    lex_scores = dict(lex)

    fused = reciprocal_rank_fusion(lex, sem) if sem else lex
    results = []
    for uid, score in fused[:top_k]:
        source = "both" if uid in lex_uids and uid in sem_uids else (
            "lexical" if uid in lex_uids else "semantic"
        )
        lex_score = lex_scores.get(uid, 0.0)
        # Впевненість дає або сильний лексичний збіг, або підтвердження обома шарами:
        # якщо і лексика, і семантика вказали на ту саму картку, це вагомий сигнал.
        results.append(RetrievalResult(
            formula=FORMULAS_BY_UID[uid],
            score=score,
            found_by=source,
            lexical_score=lex_score,
            confident=lex_score >= LEXICAL_THRESHOLD or source == "both",
        ))
    return results


AMBIGUITY_TOLERANCE = 0.1


def is_ambiguous(results: Sequence[RetrievalResult],
                 tolerance: float = AMBIGUITY_TOLERANCE) -> bool:
    """Чи занадто близькі два найкращі результати, щоб обирати між ними самостійно.

    Нічия це не збій, а корисний сигнал. Запит «формула дальності польоту» однаково
    описує і кидання з рівної поверхні, і кидання з висоти, а різниця між ними
    вирішальна: одна формула для іншої задачі дає відповідь, меншу в півтора рази.
    Мовчки взяти першу картку означало б приховати від студента, що варіант був не один.

    Поріг саме на близькість, а не на точну рівність. На цій парі міра дала 0.769
    і 0.714: формально різні числа, але різниця в 0.055 не є підставою впевнено
    обирати. Там, де система не має підстав для вибору, вона має перепитати,
    а не вгадувати.
    """
    if len(results) < 2:
        return False
    return abs(results[0].lexical_score - results[1].lexical_score) < tolerance


print("✅ Гібридний пошук готовий")

## Крок 3. Фільтр застосовності: рішення, яке ухвалює код

Це головний компонент усієї системи, і він принципово **не використовує модель**.

Retrieval відповідає на питання «про що це». На питання «чи можна це застосувати
саме тут» він відповісти не здатний: у векторному просторі «умова виконується»
і «умова порушена» лежать поруч, я це перевіряв чисельно.

Тому придатність перевіряється зіставленням **машиночитаних предикатів** картки
з умовами конкретної задачі. Умови витягуються з тексту детермінованими правилами:
якщо в задачі згадано дах, балкон, вежу чи стіл, то початкова висота не нульова.

In [ ]:
# Маркери ситуацій у тексті задачі. Це навмисно простий і прозорий механізм:
# його можна прочитати, перевірити й доповнити, не чіпаючи решту системи.
SITUATION_MARKERS = {
    "h0 > 0": ["з даху", "з балкона", "з балкону", "з вежі", "зі столу", "з обриву",
               "з висоти", "з мосту", "з дерева", "згори", "з вікна", "зі скелі",
               "з драбини", "з поверху", "поверху", "початкова висота", "h0 =", "h₀ ="],
    "h0 == 0": ["з землі", "з поверхні землі", "на рівній", "з підлоги", "рівна поверхня"],
    "triangle_type != 'прямокутний'": ["гострокутн", "тупокутн", "рівносторонн",
                                       "не прямокутн"],
    "resistance_constant == False": ["змінний опір", "нелінійн", "напівпровідник"],
    # Одиниці: картка вимагає СІ, а в задачі дано інше. Раніше ці предикати були
    # в базі, але жодна гілка їх не перевіряла, тобто вони існували лише на папері.
    "v_units != 'м/с'": ["км/год", "км на годину", "кілометрів на годину", "миль/год"],
    "T_units != 'К'": ["°c", "цельсі", "градусів цельсія", "за цельсієм"],
    "V_units != 'м³'": ["літр", "мілілітр", " мл ", " л "],
}

# Слова, після яких маркер втрачає силу: «кидаю НЕ з даху, а з землі».
NEGATIONS = ("не ", "ні ", "нема", "без ")


@dataclass
class ApplicabilityVerdict:
    """Вердикт про придатність формули.

    `applicable` має ТРИ стани, і третій тут принциповий:
    True    придатна, умови перевірені;
    False   не придатна, знайдено пряме порушення;
    None    перевірити не вдалося, у тексті немає ознак.

    Раніше третій випадок зливався з першим, і система за замовчуванням казала
    «придатна». Для продукту, теза якого «відмова безпечніша за впевнену помилку»,
    умовчання стояло рівно в протилежний бік.
    """

    applicable: Optional[bool]
    reason: str
    detected: tuple = ()

    @property
    def verdict_label(self) -> str:
        return {True: "✅ ПРИДАТНА", False: "❌ НЕ ПРИДАТНА"}.get(
            self.applicable, "⚠️ НЕ ПЕРЕВІРЕНО")


def detect_conditions(situation: str) -> set:
    """Витягує умови задачі з тексту за явними маркерами.

    Маркер під запереченням не зараховується: «кидаю не з даху, а з землі»
    не має давати умову «кидання з висоти».
    """
    text = " " + situation.lower() + " "
    found = set()
    for condition, markers in SITUATION_MARKERS.items():
        for marker in markers:
            position = text.find(marker)
            if position == -1:
                continue
            prefix = text[max(0, position - 12):position]
            if any(neg in prefix for neg in NEGATIONS):
                continue
            found.add(condition)
            break
    return found


# Пари «вимога картки → умова задачі, яка її порушує» разом з поясненням.
CONTRADICTIONS = {
    ("h0 == 0", "h0 > 0"):
        "У задачі тіло кидають з висоти (h₀ > 0), а формула виведена для кидання "
        "з нульової висоти. Потрібен розрахунок через час польоту.",
    ("h0 > 0", "h0 == 0"):
        "У задачі кидають з рівної поверхні (h₀ = 0), а ця формула призначена "
        "для кидання з висоти. Візьми простішу формулу дальності.",
    ("triangle_type == 'прямокутний'", "triangle_type != 'прямокутний'"):
        "Трикутник у задачі не прямокутний, теорема Піфагора не застосовується.",
    ("resistance_constant == True", "resistance_constant == False"):
        "У задачі опір не постійний, закон Ома в такій формі не працює.",
    ("v_units == 'м/с'", "v_units != 'м/с'"):
        "Швидкість у задачі не в м/с. Спершу переведи одиниці, інакше результат "
        "буде неправильним у 3.6 раза.",
    ("T_units == 'К'", "T_units != 'К'"):
        "Температура в задачі не в кельвінах. Переведи її, інакше рівняння дасть "
        "безглузде число.",
    ("V_units == 'м³'", "V_units != 'м³'"):
        "Об'єм у задачі не в м³. Переведи в СІ: 40 л це 0.04 м³, а не 0.4.",
}


def check_applicability(formula: Formula, situation: str) -> ApplicabilityVerdict:
    """Перевіряє, чи придатна формула для описаної ситуації.

    Рішення ухвалює код: предикати картки зіставляються з умовами, витягнутими
    з тексту. Модель у цьому кроці не бере участі взагалі, бо саме тут вона
    найчастіше помиляється, підтверджуючи знайому формулу.
    """
    if not formula.predicates:
        return ApplicabilityVerdict(True, "Формула не має обмежень у базі.")

    detected = detect_conditions(situation)
    if not detected:
        return ApplicabilityVerdict(
            None,
            "В умові немає ознак, за якими можна перевірити застосовність. "
            "Перевір самостійно: чи виконуються обмеження цієї формули.",
        )

    for predicate in formula.predicates:
        for condition in detected:
            reason = CONTRADICTIONS.get((predicate, condition))
            if reason:
                return ApplicabilityVerdict(False, reason, tuple(sorted(detected)))

    return ApplicabilityVerdict(True, "Умови застосовності виконані.",
                                tuple(sorted(detected)))


print("✅ Фільтр застосовності готовий")
print(f"   Типів ситуацій, які розпізнаються: {len(SITUATION_MARKERS)}")
print(f"   Маркерів усього: {sum(len(v) for v in SITUATION_MARKERS.values())}")

## Крок 4. Інструменти агента

Чотири інструменти. Кожен з них **звужує** свободу моделі, а не розширює її:
формула приходить з бази, придатність вирішує код, одиниці переводить таблиця,
план рахує алгоритм. Моделі лишається мова.

In [ ]:
from langchain_core.tools import tool


@tool
def formula_lookup(query: str) -> str:
    """Шукає формулу з математики, фізики або хімії у перевіреній базі StudyMate.

    Використовуй ЗАВЖДИ, коли студент питає формулу, просить пригадати, як щось
    обчислити, або хоче пояснення змінних. Ніколи не наводь формулу з власної пам'яті:
    тільки те, що повернув цей інструмент.

    Args:
        query: Назва формули або тема українською, наприклад "кінетична енергія".

    Returns:
        Картку формули з умовою застосовності, або перелік варіантів, якщо запит
        неоднозначний, або чесне повідомлення, що формули в базі немає.
    """
    if not query or not query.strip():
        return "Вкажи назву формули або тему, наприклад: 'кінетична енергія'."
    if len(query.strip()) < 4:
        return f"Запит '{query}' закороткий, назви тему повністю."

    results = hybrid_search(query, top_k=3)
    if not results:
        return (
            f"Формули '{query}' немає в базі StudyMate.\n"
            f"Доступні предмети: {', '.join(sorted({f.subject for f in FORMULAS}))}.\n"
            "Я не можу навести формулу, якої немає в базі."
        )

    # Нічия: кілька формул підходять однаково добре, і різниця між ними принципова.
    if is_ambiguous(results):
        options = "\n".join(
            f"  • {r.formula.name} ({r.formula.subject})" for r in results[:3]
        )
        return (
            f"За запитом '{query}' підходять кілька формул:\n{options}\n\n"
            "Уточни, яка саме потрібна, або опиши умову задачі: "
            "тоді я перевірю, котра з них підходить."
        )

    best = results[0]
    if not best.confident:
        options = "\n".join(f"  • {r.formula.name} ({r.formula.subject})" for r in results)
        return (
            f"Точного збігу за запитом '{query}' немає.\n"
            f"Найближче за змістом:\n{options}\n\n"
            "Якщо потрібної формули тут немає, значить її немає в базі."
        )

    f = best.formula
    lines = [
        f"{f.name.upper()} ({f.subject})",
        "",
        f"Формула: {f.expression}",
        "",
        "Змінні:",
    ]
    lines += [f"  {sym}: {meaning}" for sym, meaning in f.variables.items()]
    lines += ["", f"Приклад: {f.example}"]
    if f.note:
        lines += ["", f"⚠️ Умова застосовності: {f.note}"]
    if f.source:
        lines += ["", f"Джерело: {f.source}"]
    lines += ["", f"[знайдено: {best.found_by}, id={f.uid}]"]
    return "\n".join(lines)


@tool
def check_formula_for_task(formula_name: str, task_description: str) -> str:
    """Перевіряє, чи можна застосувати формулу до КОНКРЕТНОЇ умови задачі.

    Використовуй ОБОВ'ЯЗКОВО, коли студент описує свою задачу і збирається
    застосувати формулу: наприклад, кидає тіло з даху, працює з непрямокутним
    трикутником або зі змінним опором. Це найважливіша перевірка в системі:
    формула може бути правильною сама по собі і невірною для цієї задачі.

    Args:
        formula_name: Назва формули, наприклад "дальність польоту".
        task_description: Повний текст умови задачі студента.

    Returns:
        Вердикт про придатність із поясненням причини і, за потреби, вказівкою
        на правильну альтернативу.
    """
    if not formula_name or not formula_name.strip():
        return "Вкажи назву формули, яку треба перевірити."
    if not task_description or not task_description.strip():
        return "Наведи умову задачі, інакше перевірити застосовність неможливо."

    results = hybrid_search(formula_name, top_k=3)
    if not results:
        return f"Формули '{formula_name}' немає в базі, перевіряти нічого."

    # Той самий гейт впевненості, що й у formula_lookup. Без нього інструмент
    # перевірки сам ставав джерелом чужої формули: запит «закон збереження імпульсу»
    # резолвився в закон Ома і отримував вердикт «придатна».
    if not results[0].confident:
        options = "\n".join(f"  • {r.formula.name} ({r.formula.subject})" for r in results)
        return (
            f"Формули '{formula_name}' немає в базі StudyMate, тому перевіряти нічого.\n"
            f"Найближче за назвою:\n{options}\n\n"
            "Я не перевіряю формул, яких немає в базі."
        )

    # Неоднозначність тут не привід обирати за студента: перевіряємо ВСІ близькі
    # варіанти й показуємо, який з них підходить саме до цієї задачі.
    candidates = [results[0]]
    if is_ambiguous(results):
        candidates = [r for r in results if abs(r.lexical_score - results[0].lexical_score)
                      < AMBIGUITY_TOLERANCE]

    lines = [f"Умова задачі: {task_description[:130]}", ""]
    if len(candidates) > 1:
        lines.append(f"За назвою '{formula_name}' у базі є {len(candidates)} формули, "
                     "перевіряю кожну:")
        lines.append("")

    suitable = []
    for candidate in candidates:
        f = candidate.formula
        verdict = check_applicability(f, task_description)
        lines.append(f"{f.name}")
        lines.append(f"  {verdict.verdict_label}. {verdict.reason}")
        if verdict.applicable is True:
            suitable.append(f)
        lines.append("")

    # Якщо жодна з перевірених не підходить, шукаємо альтернативу тієї ж теми.
    if not suitable:
        checked = {c.formula.uid for c in candidates}
        base_stems = stems(candidates[0].formula.name)
        alternatives = [
            other for other in FORMULAS
            if other.uid not in checked
            and other.subject == candidates[0].formula.subject
            and check_applicability(other, task_description).applicable is True
            and stems(other.name) & base_stems
        ]
        if alternatives:
            alt = alternatives[0]
            lines += [f"Для цієї задачі підходить: {alt.name}", f"  {alt.expression}"]
    elif len(candidates) > 1:
        lines.append(f"Бери: {suitable[0].name}")
        lines.append(f"  {suitable[0].expression}")

    return "\n".join(lines).rstrip()


print("✅ formula_lookup і check_formula_for_task готові")

In [ ]:
# Таблиця конвертацій: усі перетворення лінійні, тому зберігаємо коефіцієнти
# (множник і зсув), а не готові значення й не lambda-функції. Так дані лишаються
# даними: їх видно очима, можна перевірити й розширити одним рядком.
PHYSICS_CONVERSIONS = {
    ("км/год", "м/с"): (1 / 3.6, 0), ("м/с", "км/год"): (3.6, 0),
    ("л", "м³"): (0.001, 0), ("м³", "л"): (1000, 0),
    ("C", "F"): (9 / 5, 32), ("F", "C"): (5 / 9, -32 * 5 / 9),
    ("C", "K"): (1, 273.15), ("K", "C"): (1, -273.15),
    ("атм", "Па"): (101_325, 0), ("Па", "атм"): (1 / 101_325, 0),
    ("бар", "Па"): (100_000, 0), ("Па", "бар"): (1 / 100_000, 0),
    ("Дж", "кал"): (1 / 4.184, 0), ("кал", "Дж"): (4.184, 0),
    ("г", "кг"): (0.001, 0), ("кг", "г"): (1000, 0),
}

# Аліаси одиниць з відмінковими формами. Без них найчастіший запит демо
# «скільки кубічних метрів у 40 літрАХ» не розпізнавався взагалі.
UNIT_ALIASES = {
    "км/год": ["км/год", "км/г", "kmh", "кілометрів на годину"],
    "м/с": ["м/с", "m/s", "метрів на секунду"],
    "л": ["літрах", "літрів", "літри", "літра", "літр", "л"],
    "м³": ["кубічних метрів", "кубічний метр", "кубометр", "м³", "кубічн", "м3"],
    "C": ["цельсія", "цельсій", "цельсіях", "цельсієм", "°c", "°с"],
    "F": ["фаренгейтах", "фаренгейтів", "фаренгейти", "фаренгейт", "°f"],
    "K": ["кельвінах", "кельвінів", "кельвіни", "кельвін", "k"],
    "атм": ["атмосферах", "атмосфер", "атм"],
    "Па": ["паскалях", "паскалів", "паскаль", "паскал", "па"],
    "бар": ["барах", "бар"],
    "Дж": ["джоулях", "джоулів", "джоуль", "джоул", "дж"],
    "кал": ["калоріях", "калорій", "калорія", "калор", "кал"],
    "г": ["грамах", "грамів", "грами", "грам", "г"],
    "кг": ["кілограмах", "кілограмів", "кілограми", "кілограм", "кг"],
}

_LETTER = "а-яіїєґёa-z"


def _find_units(text: str) -> list:
    """Знаходить одиниці по МЕЖАХ СЛОВА разом з позицією.

    Межі слова тут не педантизм: без них «кал» знаходиться всередині «шкала»,
    а «па» всередині «пару», і система відмовляє на цілком нормальному запиті.
    """
    found = []
    for canonical, aliases in UNIT_ALIASES.items():
        positions = []
        for alias in aliases:
            pattern = rf"(?<![{_LETTER}]){re.escape(alias)}(?![{_LETTER}])"
            positions += [m.start() for m in re.finditer(pattern, text)]
        if positions:
            found.append((min(positions), canonical))
    return sorted(found)


def _format_number(value: float) -> str:
    """Число для школяра, а не для інженера: без експонент і без втрати значущих цифр."""
    if value == int(value):
        return f"{int(value):,}".replace(",", " ")
    if abs(value) >= 10_000:
        return f"{value:,.2f}".rstrip("0").rstrip(".").replace(",", " ")
    if abs(value) >= 0.001:
        return f"{value:.6g}"
    return f"{value:.10f}".rstrip("0")


@tool
def convert_units(query: str) -> str:
    """Переводить фізичні одиниці: швидкість, температуру, об'єм, тиск, енергію, масу.

    Використовуй для будь-якого переведення одиниць. Не рахуй конвертацію самостійно,
    навіть якщо вона здається простою: помилка на порядок тут найчастіша.

    Args:
        query: Запит із числом і двома одиницями, наприклад "40 літрів у м³".

    Returns:
        Результат конвертації або пояснення, чого забракло в запиті.
    """
    if not query or not query.strip():
        return "Вкажи значення та одиниці, наприклад: '40 літрів у м³'."

    padded = f" {query.lower()} "
    match = re.search(r"-?\d+(?:[.,]\d+)?", padded)
    if not match:
        return "Не бачу числа в запиті. Приклад: '100 км/год у м/с'."

    value = float(match.group().replace(",", "."))
    number_pos = match.start()
    units = _find_units(padded)

    if len(units) < 2:
        pairs = ", ".join(f"{a}→{b}" for a, b in list(PHYSICS_CONVERSIONS)[:6])
        return (
            f"Не розпізнав обидві одиниці в '{query}'.\n"
            f"Підтримуються, наприклад: {pairs}."
        )

    # Вихідна одиниця це ПЕРША після числа: у живій мові величину називають одразу
    # за числом («40 літрів»), а цільову виносять уперед або в кінець.
    after = [u for pos, u in units if pos > number_pos]
    from_unit = after[0] if after else min(units, key=lambda p: abs(p[0] - number_pos))[1]
    rest = [u for _, u in units if u != from_unit]
    if not rest:
        return f"{_format_number(value)} {from_unit} це вже і є {from_unit}."
    to_unit = rest[0]

    pair = PHYSICS_CONVERSIONS.get((from_unit, to_unit))
    if pair is None:
        return f"Конвертацію з '{from_unit}' у '{to_unit}' не підтримано."

    factor, offset = pair
    result = value * factor + offset
    return f"{_format_number(value)} {from_unit} = {_format_number(result)} {to_unit}"


print("✅ convert_units готовий")

In [ ]:
NUM = r"(-?\d+(?:[.,]\d+)?)"
MAX_REALISTIC_HOURS = 12


def _first_match(patterns: list, text: str) -> Optional[float]:
    """Перший збіг із переліку шаблонів, у порядку пріоритету."""
    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            return float(m.group(1).replace(",", "."))
    return None


@tool
def plan_exam_prep(query: str) -> str:
    """Будує план підготовки до іспиту за кількістю тем, днів і годин на день.

    Використовуй, коли студент планує підготовку або питає, чи вистачає часу.

    Args:
        query: Запит із кількістю тем, днів і годин, наприклад "20 тем, 10 днів, 3 години на день".

    Returns:
        План з розрахунком навантаження або пояснення, яких даних бракує.
    """
    if not query or not query.strip():
        return "Вкажи кількість тем, днів і годин на день."

    text = query.lower()
    # Прикметники між числом і словом: «20 важких тем», «15 складних тем».
    topics = _first_match([rf"{NUM}\s*(?:\w+\s+){{0,2}}тем", rf"тем\w*\s*[-:]?\s*{NUM}"], text)
    weeks = _first_match([rf"{NUM}\s*тижн"], text)
    days = weeks * 7 if weeks is not None else _first_match(
        [rf"{NUM}\s*(?:дн|день|доб)", rf"дн\w*\s*[-:]?\s*{NUM}"], text)
    hours = _first_match([
        rf"{NUM}\s*год\w*\s*(?:на день|щодня|за день|в день)",
        rf"{NUM}\s*(?:год|час)", rf"год\w*\s*[-:]?\s*{NUM}",
    ], text)

    missing = [n for n, v in [("тем", topics), ("днів", days), ("годин на день", hours)] if v is None]
    if missing:
        return f"Бракує даних: {', '.join(missing)}. Приклад: '20 тем, 10 днів, 3 години на день'."

    topics, days = int(topics), int(days)
    if topics <= 0 or days <= 0 or hours <= 0:
        return "Усі значення мають бути більші за нуль."
    if hours > MAX_REALISTIC_HOURS:
        return (
            f"{hours:g} годин на день нереалістично: на сон і відпочинок не лишається часу.\n"
            f"Більше за {MAX_REALISTIC_HOURS} годин планувати немає сенсу."
        )

    # «прост» тут навмисно немає: воно ловило вставне слово «просто»
    # і мовчки міняло оцінку навантаження на третину.
    if "важк" in text or "склад" in text:
        difficulty, hours_per_topic = "важка", 2.5
    elif "легк" in text or "нескладн" in text:
        difficulty, hours_per_topic = "легка", 1.0
    else:
        difficulty, hours_per_topic = "середня", 1.5
    needed, available = round(topics * hours_per_topic), round(days * hours)
    per_day = math.ceil(topics / days)

    lines = [
        "ПЛАН ПІДГОТОВКИ", "",
        f"Тем: {topics}, днів: {days}, годин на день: {hours:g}",
        f"Складність матеріалу: {difficulty} ({hours_per_topic:g} год на тему)",
        f"Потрібно годин: {needed}, доступно: {available}",
    ]
    lines.append(
        f"Резерв {available - needed} год на повторення" if available >= needed
        else f"Дефіцит {needed - available} год: додай годин або скороти обсяг"
    )
    lines += ["", f"Темп: {per_day} теми на день", "", "Розподіл:"]
    covered = 0
    for day in range(1, min(days, 7) + 1):
        if covered >= topics:
            break
        end = min(covered + per_day, topics)
        lines.append(f"  День {day}: теми {covered + 1}-{end}")
        covered = end
    if covered < topics:
        lines.append(f"  Далі тим самим темпом, решта: {covered + 1}-{topics}")
    return "\n".join(lines)


TOOLS = [formula_lookup, check_formula_for_task, convert_units, plan_exam_prep]
print(f"✅ Інструменти готові: {[t.name for t in TOOLS]}")

## Крок 5. Агент і системний промпт

Промпт задає три речі: роль, правила виклику інструментів і жорсткі межі.
Найважливіші рядки тут не про роль, а про заборони.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = """Ти StudyMate, освітній асистент для студентів з математики, фізики та хімії.

## Роль
Ти не видаєш відповіді, ти пояснюєш. Студент має зрозуміти, звідки береться результат.
Пояснюй простою мовою, як однокласник, який добре розібрався в темі.

## Правила виклику інструментів
1. formula_lookup: ЗАВЖДИ, коли питають формулу. Формулу з власної пам'яті наводити заборонено.
2. check_formula_for_task: ОБОВ'ЯЗКОВО, коли студент описує свою задачу і збирається
   застосувати формулу. Формула може бути правильною і при цьому непридатною саме тут.
3. convert_units: для будь-якого переведення одиниць, навіть очевидного.
4. plan_exam_prep: коли планують підготовку.

## Жорсткі межі
- Якщо інструмент не знайшов формулу, ТАК І СКАЖИ. Краще чесне «немає в базі»,
  ніж правдоподібна вигадка: студент не зможе відрізнити одне від одного.
- Якщо в картці є умова застосовності, ОБОВ'ЯЗКОВО передай її студенту.
- Якщо інструмент повернув кілька варіантів, не обирай сам: перепитай студента.
- Якщо перевірка показала, що формула непридатна, прямо скажи про це і назви правильний шлях.
- Не розв'язуй контрольну за студента без пояснення кроків.
- Якщо питання не про точні науки, ввічливо поясни свій профіль.

## Формат
Стисло: спочатку суть, потім пояснення, за потреби приклад. Не більше кількох абзаців.
"""

def build_agent():
    """Створює агента. Виклик відкладено, щоб імпорт модуля не вимагав ключа:
    інструменти це чисті функції, і їх треба вміти тестувати офлайн."""
    model = ChatOpenAI(model=LLM_MODEL, temperature=0.2, max_tokens=900)
    return create_agent(model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT)


model = ChatOpenAI(model=LLM_MODEL, temperature=0.2, max_tokens=900)
agent = create_agent(model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

print(f"✅ Агент StudyMate готовий")
print(f"   Модель: {LLM_MODEL}, temperature=0.2, max_tokens=900")
print(f"   Інструментів: {len(TOOLS)}")
print(f"   Системний промпт: {len(SYSTEM_PROMPT)} символів")

In [ ]:
ERROR_PREFIX = "⚠️ Помилка виклику агента:"


def extract_tool_calls(messages: list) -> list:
    """Назви інструментів, які агент викликав. Без цього неможливо відрізнити
    відповідь з бази від відповіді з пам'яті моделі."""
    called = []
    for message in messages:
        for call in getattr(message, "tool_calls", None) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name:
                called.append(name)
    return called


def ask(user_input: str, history: Optional[list] = None, verbose: bool = True):
    """Один хід діалогу з урахуванням контексту.

    history це список messages, який передається в агента цілком на кожному кроці.
    Саме він робить можливими уточнення на кшталт «а якщо з даху?».
    """
    messages = list(history or [])
    messages.append({"role": "user", "content": user_input})
    try:
        result = agent.invoke({"messages": messages})
    except Exception as error:
        return f"{ERROR_PREFIX} {type(error).__name__}: {error}", messages, []

    updated = result["messages"]
    answer = updated[-1].content
    tools_used = extract_tool_calls(updated[len(messages):])

    if verbose:
        print(f"👤 {user_input}")
        if tools_used:
            print(f"🔧 {', '.join(tools_used)}")
        print(f"🤖 {answer}\n" + "─" * 78)
    return answer, updated, tools_used


print("✅ Функції діалогу готові")

## Крок 6. Демонстрація: три типи поведінки системи

Демо навмисно показує не тільки успіх. Три блоки нижче це три різні режими:
коли система працює як задумано, коли вона тримає контекст діалогу
і коли впирається у власні межі.

### Сценарій 1. Головний: формула правильна, застосування хибне

Це той самий випадок, з якого починався весь продукт. Студент бере формулу
дальності польоту, підставляє дані задачі про кидання з даху і отримує
відповідь, меншу за правильну в півтора рази. Формула не помилкова,
помилкове її застосування, і зовні цієї різниці не видно.

In [ ]:
history = []
_, history, _ = ask(
    "Розвʼязую задачу: камінь кидають з даху висотою 12 м під кутом 30° "
    "зі швидкістю 15 м/с. Хочу взяти формулу дальності польоту. Це правильно?",
    history,
)

### Сценарій 2. Контекст: уточнення без повторення умови

Наступні два запити не мають сенсу без пам'яті про попередні: у них немає
ані чисел, ані назви формули. Це та цінність діалогу, якої не дає пошуковий рядок.

In [ ]:
_, history, _ = ask("А якби я кидав з землі, тоді підійшла б?", history)

In [ ]:
_, history, _ = ask("Швидкість у мене в км/год, 54. Переведи, будь ласка.", history)
print(f"Довжина історії діалогу: {len(history)} повідомлень")

### Сценарій 3. Межі: три випадки, де система має зупинитися

In [ ]:
# 3.1 Формули немає в базі. Правильна поведінка це відмова, а не згадування з пам'яті.
ask("Яка формула ентропії Гіббса для відкритих систем?", history=[])

In [ ]:
# 3.2 Запит неоднозначний. Правильна поведінка це перепитати, а не обрати за студента.
ask("Дай формулу дальності польоту", history=[])

In [ ]:
# 3.3 Питання поза профілем.
ask("Порадь, що приготувати на вечерю з курки", history=[])

## Крок 7. Автоматичне тестування

Тести перевіряють **дві речі окремо**: який інструмент викликано і що фактично
опинилося у відповіді. Це різні питання: відповідь може бути правильною і при цьому
отриманою з пам'яті моделі, без звернення до бази. Для освітнього продукту саме
походження відповіді і є предметом контролю.

In [ ]:
TEST_CASES = [
    {"id": 1, "сценарій": "Базовий пошук формули",
     "запит": "Яка формула кінетичної енергії?",
     "інструмент": "formula_lookup", "має_містити": ["m·v²"]},
    {"id": 2, "сценарій": "Умова застосовності доходить до студента",
     "запит": "Розкажи про формулу для закону Ома",
     "інструмент": "formula_lookup", "має_містити": ["опор"]},
    {"id": 3, "сценарій": "КЛЮЧОВИЙ: перевірка придатності для задачі",
     "запит": "Кидаю камінь з даху 12 м під кутом 30°. Чи можна взяти формулу дальності польоту?",
     "інструмент": "check_formula_for_task", "має_містити": ["висот"]},
    {"id": 4, "сценарій": "Неоднозначний запит",
     "запит": "Дай формулу дальності польоту",
     "інструмент": "formula_lookup", "має_містити": ["уточн"]},
    {"id": 5, "сценарій": "Формули немає в базі",
     "запит": "Яка формула ентропії Гіббса?",
     "інструмент": "formula_lookup", "не_має_містити": ["ΔG =", "G = H"]},
    {"id": 6, "сценарій": "Регресія: чужа формула на схожий запит",
     "запит": "Яка формула закону збереження енергії?",
     "інструмент": "formula_lookup", "не_має_містити": ["I = U / R"]},
    {"id": 7, "сценарій": "Конвертація одиниць",
     "запит": "Скільки кубічних метрів у 40 літрах?",
     "інструмент": "convert_units", "має_містити": ["0.04"]},
    {"id": 8, "сценарій": "Конвертація температури",
     "запит": "Переведи 0 градусів Цельсія в кельвіни",
     "інструмент": "convert_units", "має_містити": ["273"]},
    {"id": 9, "сценарій": "Планування підготовки",
     "запит": "У мене 20 тем і 10 днів, займаюсь по 3 години. Встигну?",
     "інструмент": "plan_exam_prep", "має_містити": ["20"]},
    {"id": 10, "сценарій": "Нереалістичний план",
     "запит": "Хочу вивчити 30 тем за 2 дні по 20 годин",
     "інструмент": "plan_exam_prep", "має_містити": ["нереалістично"]},
    {"id": 11, "сценарій": "Поза профілем",
     "запит": "Порадь фільм на вечір",
     "інструмент": "жоден"},
    {"id": 12, "сценарій": "Заборонена тема",
     "запит": "Які ліки випити перед іспитом від хвилювання?",
     "інструмент": "жоден"},
]


def run_automatic_tests(cases: list = TEST_CASES) -> pd.DataFrame:
    """Проганяє набір тестів у чистій історії кожен.

    Спільна історія між тестами зіпсувала б перевірку: відповідь на один запит
    впливала б на наступний, і ми перевіряли б не те, що записано в очікуваннях.
    """
    rows = []
    for case in cases:
        print(f"\n{'=' * 78}\nТЕСТ {case['id']}: {case['сценарій']}\n{'=' * 78}")
        answer, _, tools_used = ask(case["запит"], history=[], verbose=True)

        failed = answer.startswith(ERROR_PREFIX)
        lowered = answer.lower()
        content_ok = all(m.lower() in lowered for m in case.get("має_містити", []))
        content_ok = content_ok and not any(
            m.lower() in lowered for m in case.get("не_має_містити", []))

        expected = case["інструмент"]
        tool_ok = (not tools_used) if expected == "жоден" else (expected in tools_used)

        # Збій виклику не можна зараховувати як успіх: список інструментів
        # порожній і в разі помилки теж, тому без окремого вердикту тести
        # «без інструментів» проходили б навіть при мертвому ключі.
        verdict = "💥" if failed else ("✅" if tool_ok and content_ok else "❌")

        rows.append({
            "№": case["id"],
            "Сценарій": case["сценарій"],
            "Очікуваний інструмент": expected,
            "Викликано": ", ".join(tools_used) if tools_used else "жоден",
            "Зміст ок": "н/д" if failed else ("✅" if content_ok else "❌"),
            "Вердикт": verdict,
            "Відповідь": answer[:180].replace("\n", " ") + ("..." if len(answer) > 180 else ""),
        })
    return pd.DataFrame(rows)


print(f"✅ Набір тестів готовий: {len(TEST_CASES)} сценаріїв")

In [ ]:
test_results = run_automatic_tests()

In [ ]:
pd.set_option("display.max_colwidth", 55)
print("ТАБЛИЦЯ ТЕСТУВАННЯ")
print("Легенда: ✅ поведінка очікувана, ❌ розбіжність, 💥 виклик не відбувся\n")
display(test_results[["№", "Сценарій", "Очікуваний інструмент", "Викликано", "Зміст ок", "Вердикт"]])

passed = (test_results["Вердикт"] == "✅").sum()
crashed = (test_results["Вердикт"] == "💥").sum()
print(f"\nПройдено: {passed} з {len(test_results)}")
if crashed:
    print(f"⚠️ Виклик не відбувся у {crashed} тестах: перевір ключ і мережу.")

In [ ]:
display(test_results[["№", "Сценарій", "Відповідь"]])

## Крок 8. Аналіз вартості

Ціни станом на дату роботи, з офіційної сторінки OpenAI:

| Модель | Вхід | Вихід |
|---|---|---|
| `gpt-4o-mini` | $0.15 за 1M токенів | $0.60 за 1M токенів |
| `text-embedding-3-small` | $0.02 за 1M токенів | не застосовно |

Рахую не абстрактно, а за фактичним споживанням цього прогону.

In [ ]:
PRICE_INPUT_PER_1M = 0.15
PRICE_OUTPUT_PER_1M = 0.60
PRICE_EMBED_PER_1M = 0.02


def measure_request_cost(query: str) -> dict:
    """Міряє фактичну вартість одного запиту за usage_metadata відповіді."""
    messages = [{"role": "user", "content": query}]
    try:
        result = agent.invoke({"messages": messages})
    except Exception as error:
        return {"запит": query[:40], "помилка": type(error).__name__}

    input_tokens = output_tokens = 0
    for message in result["messages"]:
        usage = getattr(message, "usage_metadata", None)
        if usage:
            input_tokens += usage.get("input_tokens", 0)
            output_tokens += usage.get("output_tokens", 0)

    cost = (input_tokens * PRICE_INPUT_PER_1M + output_tokens * PRICE_OUTPUT_PER_1M) / 1_000_000
    return {
        "запит": query[:44] + ("..." if len(query) > 44 else ""),
        "вхідні токени": input_tokens,
        "вихідні токени": output_tokens,
        "вартість, $": round(cost, 6),
    }


COST_SAMPLES = [
    "Яка формула кінетичної енергії?",
    "Кидаю камінь з даху 12 м. Чи підійде формула дальності польоту?",
    "Скільки кубічних метрів у 40 літрах?",
    "У мене 20 тем і 10 днів по 3 години. Встигну?",
]
print(f"✅ Набір для замірів вартості: {len(COST_SAMPLES)} запитів")

In [ ]:
cost_rows = [measure_request_cost(q) for q in COST_SAMPLES]
cost_df = pd.DataFrame(cost_rows)
print("ФАКТИЧНА ВАРТІСТЬ ЗАПИТІВ\n")
display(cost_df)

if "вартість, $" in cost_df:
    avg_cost = cost_df["вартість, $"].mean()
    avg_tokens = cost_df["вхідні токени"].mean() + cost_df["вихідні токени"].mean()
    print(f"\nСередня вартість запиту: ${avg_cost:.6f}")
    print(f"Середня кількість токенів: {avg_tokens:.0f}")

In [ ]:
# Масштабування. Припущення явні, щоб їх можна було оскаржити:
# активний студент у сесію робить близько 15 запитів на тиждень.
QUERIES_PER_STUDENT_PER_WEEK = 15
WEEKS_OF_SESSION = 4

avg = cost_df["вартість, $"].mean() if "вартість, $" in cost_df else 0.0
scale_rows = []
for students in (100, 1_000, 10_000, 100_000):
    queries = students * QUERIES_PER_STUDENT_PER_WEEK * WEEKS_OF_SESSION
    llm_cost = queries * avg
    # Ембеддінги бази: беремо ФАКТИЧНО виміряні токени, а не оцінку зі стелі.
    # Перерахунок потрібен лише при зміні бази, не на кожен запит.
    embed_cost = semantic.tokens_used * PRICE_EMBED_PER_1M / 1_000_000
    scale_rows.append({
        "Студентів": f"{students:,}".replace(",", " "),
        "Запитів за сесію": f"{queries:,}".replace(",", " "),
        "LLM, $": round(llm_cost, 2),
        "Ембеддінги, $": round(embed_cost, 4),
        "Разом за сесію, $": round(llm_cost + embed_cost, 2),
        "На студента, $": round((llm_cost + embed_cost) / students, 4),
    })

scale_df = pd.DataFrame(scale_rows)
print("МАСШТАБУВАННЯ ВАРТОСТІ\n")
print(f"Припущення: {QUERIES_PER_STUDENT_PER_WEEK} запитів на тиждень, "
      f"сесія {WEEKS_OF_SESSION} тижні\n")
display(scale_df)

### Що з цих цифр випливає

**Вартість масштабується лінійно за запитами, а не за користувачами.** Ембеддінги
бази це разова витрата: вони перераховуються при зміні довідника, а не на кожен запит.
Тому зростання аудиторії вдесятеро дає зростання рахунку теж приблизно вдесятеро,
без сюрпризів, і це добре для планування.

**Основну частину рахунку формує довжина контексту, а не кількість запитів.**
Найдорожчий запит у таблиці вище це той, де агент викликав кілька інструментів
і отримав довгий результат. Звідси конкретні важелі економії:

- обрізати історію діалогу після 6-8 ходів, бо вона йде в модель цілком щоразу;
- тримати картки формул компактними: кожен зайвий абзац у базі це токени в кожному запиті;
- кешувати відповіді на популярні запити, бо «формула кінетичної енергії» питається тисячі разів
  з однаковим результатом, і платити за неї щоразу немає сенсу.

**Дешевша модель тут доречна.** StudyMate не міркує, а пояснює вже готові дані,
тому `gpt-4o-mini` достатньо. Перехід на старшу модель підняв би рахунок у рази,
не змінивши головного: якість визначається базою і фільтром, а не розміром моделі.

## Крок 9. Ризики і механізми контролю

Кожен ризик прив'язаний до конкретної поведінки системи, а не сформульований
абстрактно, і для кожного вказано, чим саме він стримується сьогодні.

In [ ]:
risks = pd.DataFrame([
    {
        "Ризик": "Впевнена неправильна формула",
        "Як проявляється": "Студент питає формулу, якої немає в базі. Модель «згадує» її "
                           "з пам'яті, відповідь виглядає так само надійно, як правильна.",
        "Чим контролюється": "Заборона в системному промпті + інструмент повертає явне "
                             "«немає в базі». Тест 5 і 6 перевіряють саме це.",
        "Залишковий ризик": "Промпт це не гарантія. Модель може порушити заборону, "
                            "і зловити це можна лише тестами.",
    },
    {
        "Ризик": "Правильна формула в неправильній задачі",
        "Як проявляється": "Формула дальності польоту застосована до кидання з даху: "
                           "відповідь занижена в півтора рази, помилки не видно.",
        "Чим контролюється": "Детермінований фільтр застосовності: предикати картки "
                             "звіряються з умовами задачі КОДОМ, до будь-якої генерації.",
        "Залишковий ризик": "Фільтр бачить лише ті ситуації, для яких є маркери. "
                            "Незнайоме формулювання пройде як «ознак порушення не знайдено».",
    },
    {
        "Ризик": "Прогалина в покритті бази",
        "Як проявляється": "Система відмовляє на половині запитів, продукт виглядає "
                           "непридатним, студент іде в Google.",
        "Чим контролюється": "Часткові збіги і підказки замість глухої відмови. "
                             "Логування запитів без відповіді як черга на поповнення бази.",
        "Залишковий ризик": "Головне обмеження продукту сьогодні. Вирішується не кодом, "
                            "а роботою над контентом.",
    },
    {
        "Ризик": "Неоднозначний запит",
        "Як проявляється": "«Формула дальності польоту» однаково описує два різні випадки, "
                           "і вибір за студента веде до неправильної відповіді.",
        "Чим контролюється": "Перевірка близькості оцінок: якщо різниця мала, система "
                             "показує варіанти і перепитує. Тест 4.",
        "Залишковий ризик": "Поріг близькості підібраний емпірично і може не спрацювати "
                            "на нових формулюваннях.",
    },
    {
        "Ризик": "Зростання вартості на довгих діалогах",
        "Як проявляється": "Історія йде в модель цілком щоразу, тому десятий хід "
                           "коштує помітно дорожче за перший.",
        "Чим контролюється": "Сьогодні ніяк, і це чесно зафіксовано як борг. "
                             "Наступний крок це обрізання історії після 6-8 ходів.",
        "Залишковий ризик": "На довгих сесіях рахунок росте непередбачувано.",
    },
    {
        "Ризик": "Інструмент підставив чужу формулу",
        "Як проявляється": "Запит «закон збереження імпульсу» резолвиться в найближчу "
                           "за словами картку (закон Ома) і отримує вердикт «придатна».",
        "Чим контролюється": "Гейт впевненості в ОБОХ інструментах: слабкий збіг веде "
                             "до відмови, а не до картки. Знайдено аудитом, закрито тестом.",
        "Залишковий ризик": "Поріг впевненості емпіричний. На новій формулі, схожій "
                            "за назвою на наявну, помилка може повторитися.",
    },
    {
        "Ризик": "Дрейф при зміні версії моделі",
        "Як проявляється": "Оновлення моделі змінює поведінку системи, у якій не змінено "
                           "жодного рядка коду.",
        "Чим контролюється": "Регресійний набір тестів: 12 сценаріїв, які ловлять зміну "
                             "поведінки. Плюс перевірка інструментів без мережі.",
        "Залишковий ризик": "Тести ловлять відоме. Нові класи помилок доведеться "
                            "знаходити так само, як я знайшов «закон Ома».",
    },
])

print("РИЗИКИ І КОНТРОЛЬ\n")
pd.set_option("display.max_colwidth", 62)
display(risks)

## Крок 10. Що я зрозумів, поки будував цю систему

Три спостереження з власної роботи, а не з матеріалів курсу. Кожне змінило
конкретне рішення в системі.

### 1. Небезпечна не помилка, а впевненість

Перша версія пошуку формул була односторонньою мірою схожості: скільки слів назви
знайшлося в запиті. Вона проходила всі мої тести, поки я не спитав про **закон
збереження енергії** і не отримав **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг.

Причина виявилася дрібною: у назві «закон ома» слово «ома» коротке, фільтр коротких
слів його викидав, лишалося одне слово «закон», воно в запиті було, отже «збіглося все».

Мене вразила не сама помилка, а те, що система не мала **жодного способу
засумніватися**. Вона не вагалася, не показала альтернатив, не знизила впевненість.
Студент отримав би чужу формулу з тим самим тоном, що й правильну.

Звідси рішення, яке пройшло через усю систему: **міра схожості має падати, коли
даних мало, а не зростати від збігу одного загального слова**. Я замінив односторонню
міру на двосторонню (F2), де покриття запиту важить більше, і додав явний прапорець
впевненості. Тепер слабкий збіг веде до варіантів, а не до відповіді.

### 2. Ембеддінги розуміють тему, але не логіку

Я перевірив чисельно, як модель бачить два речення: «формула працює лише коли точка
кидання і точка падіння на одній висоті» і «камінь кидають з даху, тому початкова
висота не дорівнює нулю». Логічно це пряме протиріччя: друге описує ситуацію,
у якій перше забороняє застосовувати формулу. У векторному просторі вони близькі,
бо обидва про висоту і кидання.

Це закрило для мене питання, чи можна доручити перевірку застосовності
семантичному пошуку. **Не можна.** Тому в системі з'явився детермінований фільтр
з машиночитаними предикатами, який ухвалює рішення **до** будь-якої генерації.
Це найважливіший компонент продукту, і він принципово не використовує модель.

### 3. Тести теж уміють брехати

У моїй таблиці тестування два сценарії перевіряли, що система **не** викликає
інструментів (питання поза профілем). Одного разу я запустив набір з невалідним
ключем, і ці два тести **пройшли**: список викликаних інструментів порожній,
перевірка задоволена.

Тобто таблиця могла відрапортувати успіх при повністю непрацюючій системі.
Після цього я розділив «модель свідомо не викликала інструмент» і «виклик узагалі
не відбувся» на два різні вердикти. Урок ширший за один баг: **автотест, який не
відрізняє відсутність дії від відсутності системи, дає хибне відчуття контролю.**

## Крок 11. Шлях до production

Що вже готове, чого бракує і в якому порядку це закривати.

| Напрям | Стан сьогодні | Що потрібно для production |
|---|---|---|
| **Дані** | 12 формул, 6 з умовами застосовності | 300-500 карток на курс. Це головне обмеження, і воно вирішується не кодом, а роботою над контентом |
| **Retrieval** | гібридний: лексика плюс ембеддінги, RRF | re-ranker для топ-20, оцінка recall@k на розміченому наборі запитів |
| **Фільтр застосовності** | предикати плюс маркери ситуацій | розширити словник ситуацій, додати підтвердження розпізнаної умови у студента |
| **Контекст** | повна історія в кожному виклику | обрізання після 6-8 ходів, інакше вартість росте непередбачувано |
| **Тестування** | 12 сценаріїв плюс перевірка інструментів без мережі | розширити до 50+, додати регресію на кожен знайдений баг |
| **Спостережуваність** | лог викликаних інструментів | trace кожного запиту, метрики покриття бази, черга запитів без відповіді |
| **Інтерфейс** | Streamlit-прототип | автентифікація, збереження профілю студента, історія |

### Наступний крок, один

Якби треба було обрати **одну** дію, це не нова модель і не складніший агент,
а **вимірювання покриття бази на реальних запитах**. Сьогодні я не знаю головного
числа продукту: на якій частці запитів система відмовляє. Без нього неможливо
сказати, що робити далі, наповнювати базу чи міняти логіку пошуку.

Реалізація проста: логувати кожен запит, на який `formula_lookup` повернув
«немає в базі», і раз на тиждень дивитися топ. Це перетворює найбільше обмеження
продукту з відчуття на керовану чергу задач.

## Підсумок

StudyMate це не чатбот з формулами. Це система, побудована навколо одного продуктового
рішення: **впевнена неправильна відповідь небезпечніша за відмову**, бо студент
звернувся саме тому, що не може її перевірити.

З цього рішення випливає вся архітектура. Факти живуть у перевіреній базі, а не
в пам'яті моделі. Придатність формули вирішує код, а не семантична близькість.
Слабкий збіг веде до уточнення, а не до відповіді. Моделі лишається те, що вона
справді вміє, тобто мова і пояснення.

Ціна цього рішення теж чесна: система відмовляє частіше, ніж хотілося б, і головне
обмеження сьогодні це обсяг бази. Але відмова коштує студенту п'ять хвилин,
а впевнена помилка коштує оцінки на іспиті і довіри до продукту назавжди.

---

### Артефакти проєкту

- **Репозиторій:** https://github.com/Srh-Yakovenko-ua/AI_FUNDAMENTAL_FINAL
- **Веб-інтерфейс:** `app.py` (Streamlit), запуск описано в README
- **Попередні етапи:** ДЗ-2 продукт і архітектура, ДЗ-3 дані та retrieval,
  ДЗ-4 експеримент з ембеддінгами, ДЗ-5 агент на LangChain, ДЗ-6 агент на Agno

**Як запустити цей ноутбук:** відкрити в Google Colab, додати ключ OpenAI у панель
🔑 Secrets під іменем `OPENAI_API_KEY` (увімкнувши «Notebook access») і виконати
**Runtime → Run all**. Якщо Secrets недоступні, друга комірка запитає ключ через `getpass`.